# Tarrant CAD — PropertyData_R_2025 ETL

Exploration / processing notebook for the Tarrant Appraisal District residential export `PropertyData_R_2025.txt`.

**Source format** (checked on the 2025 file):
- Pipe-delimited (`|`), CRLF line endings, 56 columns, ~659K rows
- Values are space-padded fixed-width inside the pipes (e.g. `|      110000|`)
- One row per account (`Record_Type = AAAA`)
- Sentinel values: deed date `12-31-1900`, deed book/page `0000000`, instrument `0000000000`, year built `0`

Steps: **Load raw → Clean → Type → Profile → Validate → Export (bronze)**.

## 1. Setup

In [2]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR = Path("../Data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_FILE = DATA_DIR / "PropertyDataSupplemental_R_2025.txt"
print(INPUT_FILE.resolve(), f"{INPUT_FILE.stat().st_size / 1e6:,.1f} MB")

/Users/user2/PTP/ptp-data/tarrant/Data/PropertyDataSupplemental_R_2025.txt 352.9 MB


## 2. Load raw
Read everything as string first so nothing (leading zeros in `Account_Num`, zips, codes) gets mangled by type inference.

In [ ]:
def show_bad(bad_line):
    print("BAD LINE:", "|".join(bad_line))
    return None  # skip the row

raw = pd.read_csv(
    INPUT_FILE,
    sep="|",
    dtype=str,
    keep_default_na=False,   # keep blanks as "" until we clean them ourselves
    encoding="latin-1",      # safe for any stray non-UTF-8 bytes
    quoting=3,               # csv.QUOTE_NONE: owner names / legal descriptions can contain quotes
    engine="python",       # required for a callable on_bad_lines
    on_bad_lines=show_bad,

)
raw.columns = raw.columns.str.strip()
print(raw.shape)
raw.head()

(658409, 28)


,RP,AppraisalYear,AccountNumber,AgentName,AgentNumber,Condition,DepreciationPercent,Latitude,Longitude,EYOC,NBHD,Quality,SiteName,SiteClassCd,SiteClassDescr,PartyId,LastSaleDate,SiteId,SitusCity,NAICSDescr,NAICSCd,PercentComplete,NoticeValue,SubdivisionName,NumberofStories,LandUseCode,Country,AddressType
0,R,2025,00001309,,,Good ...,,32.7591221347,-97.3281275822,1995,4T050C ...,Average ...,FORT WORTH ORIGINAL TOWN-29-32 ...,A1,Residential - Single Family ...,3616171,2016-08-22,100001309,,,,100%,,FORT WORTH ORIGINAL TOWN,2,Residential 4T050 ...,UNITED STATES OF AMERICA,Standard
1,R,2025,00006483,,,Fair ...,,32.762335283,-97.3271676957,1955,2M210D ...,Low ...,FORT WORTH ORIGINAL TOWN 124 1B,A1,Residential - Single Family ...,59570,1900-01-01,80874275,,,,100%,101114,FORT WORTH ORIGINAL TOWN,2,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard
2,R,2025,00006491,,,Poor ...,,32.7622310968,-97.3271440167,1945,2M210D ...,Low ...,FORT WORTH ORIGINAL TOWN 124 1C1,A1,Residential - Single Family ...,775725,1990-06-01,80874276,,,,100%,60890,FORT WORTH ORIGINAL TOWN,1,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard
3,R,2025,00006505,OCONNOR & ASSOCIATES,00436,,,32.7621419405,-97.3270950408,,2M210D ...,,FORT WORTH ORIGINAL TOWN Block 124 Lot 1D,C1,Residential - Vacant Land ...,11922,1900-01-01,80874289,,,,0%,,FORT WORTH ORIGINAL TOWN,0,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard
4,R,2025,00006513,OCONNOR & ASSOCIATES,00436,Fair ...,,32.7619349338,-97.326950484,1955,2M210D ...,Low ...,FORT WORTH ORIGINAL TOWN 124 1F,A1,Residential - Single Family ...,11923,1984-03-10,80874277,,,,100%,141751,FORT WORTH ORIGINAL TOWN,1,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard


In [4]:
raw.describe(include="all").T

,count,unique,top,freq
RP,658409,1,R,658409
AppraisalYear,658409,1,2025,658409
AccountNumber,658409,658409,00001309,1
AgentName,658409,1092,,489358
AgentNumber,658409,1102,,489358
Condition,658409,7,Average ...,232793
DepreciationPercent,658409,1,,658409
Latitude,658409,632356,00000000000000000000000000000000,529
Longitude,658409,632313,00000000000000000000000000000000,529
EYOC,658409,104,1975,91138


In [5]:
raw.to_csv(OUTPUT_DIR / "PropertyDataSupplemental_R_2025.csv", index=False)